# Modelo de fraude

Entrenamiento con POO, busqueda de hiperparametros y exportacion de modelos a `artifact/`.

In [56]:
from __future__ import annotations

from pathlib import Path

from sklearn.metrics import classification_report

In [57]:
import sys
import importlib

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_PROCESSED = ROOT / "data" / "processed"
ARTIFACT = ROOT / "artifact"
ARTIFACT.mkdir(parents=True, exist_ok=True)

import scripts.features.build_features as build_features_module

build_features_module = importlib.reload(build_features_module)

build_features = build_features_module.build_features

import scripts.models.fraud_model as fraud_model_module

fraud_model_module = importlib.reload(fraud_model_module)

from scripts.models.fraud_model import (
    FraudModelTrainer,
    build_predictions,
    build_preprocess,
    default_model_configs,
    export_models,
    prepare_training_data,
    split_training_data,
    train_models,
)

In [58]:
df = build_features()
df.head()

,id_siniestro,id_poliza,id_asegurado,id_proveedor,ramo,cobertura,fecha_ocurrencia,fecha_reporte,monto_reclamado,monto_estimado,...,score_freq_conductor,score_rc_only,score_proveedor,score_docs_incompletos,score_docs_inconsistentes,score_dinamica_sospechosa,score_sin_tercero,score_reporte_tardio,score_monto_suma_asegurada,score_total_reglas
0,1,1,50,28.0,OTRO,Cobertura especial,2026-05-11,2026-05-27,4830.07,4173.47,...,0,0,5,0,10,0,0,5,0,24
1,2,2,8,11.0,HOGAR,Responsabilidad familiar,2025-09-17,2025-09-27,2482.29,1985.12,...,0,0,10,0,0,0,0,5,0,15
2,3,3,56,15.0,HOGAR,Robo domiciliario,2023-11-23,2023-12-09,39825.18,39104.68,...,0,0,5,4,10,0,0,5,4,36
3,4,4,13,4.0,SALUD,Hospitalización,2024-01-20,2024-01-22,27810.18,24813.58,...,0,0,5,0,0,0,0,0,4,9
4,5,5,56,26.0,VIDA,Enfermedad grave,2023-09-24,2023-10-05,4730.35,4924.49,...,0,0,0,0,0,0,0,5,0,5


In [59]:
X, y, cat_cols, num_cols = prepare_training_data(df)
X.head()

,ramo,cobertura,monto_reclamado,monto_estimado,monto_pagado,estado,sucursal,documentos_completos,beneficiario,dias_desde_inicio_poliza,...,score_freq_conductor,score_rc_only,score_proveedor,score_docs_incompletos,score_docs_inconsistentes,score_dinamica_sospechosa,score_sin_tercero,score_reporte_tardio,score_monto_suma_asegurada,score_total_reglas
0,OTRO,Cobertura especial,4830.07,4173.47,0.00,CIERRE_SIN_CONSECUENCIA,Sucursal Manta,True,CLINICA,348,...,0,0,5,0,10,0,0,5,0,24
1,HOGAR,Responsabilidad familiar,2482.29,1985.12,1540.39,PAGO_TOTAL,Matriz Guayaquil,True,PERITO,303,...,0,0,10,0,0,0,0,5,0,15
2,HOGAR,Robo domiciliario,39825.18,39104.68,38351.48,PAGO_TOTAL,Sucursal Manta,False,PERITO,275,...,0,0,5,4,10,0,0,5,4,36
3,SALUD,Hospitalización,27810.18,24813.58,0.00,NEGATIVA,Sucursal Cuenca,True,TALLER,189,...,0,0,5,0,0,0,0,0,4,9
4,VIDA,Enfermedad grave,4730.35,4924.49,381.04,ANTICIPO,Sucursal Cuenca,True,CLINICA,134,...,0,0,0,0,0,0,0,5,0,5


In [60]:
X_train, X_test, y_train, y_test = split_training_data(
    X,
    y,
    test_size=0.2,
    random_state=42,
 )

X_train.shape, X_test.shape


((80, 70), (20, 70))

In [61]:
preprocess = build_preprocess(cat_cols, num_cols)
preprocess

,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'most_frequent'
,fill_value,None


In [62]:
configs = default_model_configs()
trainer = FraudModelTrainer(preprocess=preprocess, random_state=42)
trained_models, results_df = train_models(trainer, configs, X_train, y_train)
results_df

,model,best_score_cv_f1,best_params
0,logistic_regression,0.920000,"{'model__solver': 'lbfgs', 'model__C': 2.15443..."
1,decision_tree,1.000000,"{'model__min_samples_split': 2, 'model__min_sa..."
2,random_forest,0.593333,"{'model__n_estimators': 200, 'model__min_sampl..."


In [63]:
saved_paths = export_models(trained_models, ARTIFACT)
saved_paths

[WindowsPath('C:/Users/H P/Desktop/hackAIthon/fraudIA-Novo-2S1R/artifact/logistic_regression.pkl'),
 WindowsPath('C:/Users/H P/Desktop/hackAIthon/fraudIA-Novo-2S1R/artifact/decision_tree.pkl'),
 WindowsPath('C:/Users/H P/Desktop/hackAIthon/fraudIA-Novo-2S1R/artifact/random_forest.pkl')]

In [64]:
preds = build_predictions(trained_models, X_test, y_test)
preds.to_csv(DATA_PROCESSED / "predictions.csv", index=False)
preds.head()

,model,y_true,y_pred,y_prob
0,logistic_regression,0,0,1.468401e-55
1,logistic_regression,0,0,0.000000e+00
2,logistic_regression,0,0,8.539582e-282
3,logistic_regression,1,1,1.000000e+00
4,logistic_regression,0,0,8.649274e-230


In [65]:
results_df

,model,best_score_cv_f1,best_params
0,logistic_regression,0.920000,"{'model__solver': 'lbfgs', 'model__C': 2.15443..."
1,decision_tree,1.000000,"{'model__min_samples_split': 2, 'model__min_sa..."
2,random_forest,0.593333,"{'model__n_estimators': 200, 'model__min_sampl..."


In [66]:
preds.groupby("model")[["y_pred", "y_prob"]].agg(["mean", "min", "max"])

y_pred            y_prob                    
                      mean min max      mean       min       max
model                                                           
decision_tree         0.15   0   1  0.150000  0.000000  1.000000
logistic_regression   0.10   0   1  0.100000  0.000000  1.000000
random_forest         0.10   0   1  0.307664  0.136843  0.645445

In [67]:
for name, model in trained_models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred, digits=3))

=== logistic_regression ===
              precision    recall  f1-score   support

           0      0.944     1.000     0.971        17
           1      1.000     0.667     0.800         3

    accuracy                          0.950        20
   macro avg      0.972     0.833     0.886        20
weighted avg      0.953     0.950     0.946        20

=== decision_tree ===
              precision    recall  f1-score   support

           0      1.000     1.000     1.000        17
           1      1.000     1.000     1.000         3

    accuracy                          1.000        20
   macro avg      1.000     1.000     1.000        20
weighted avg      1.000     1.000     1.000        20

=== random_forest ===
              precision    recall  f1-score   support

           0      0.944     1.000     0.971        17
           1      1.000     0.667     0.800         3

    accuracy                          0.950        20
   macro avg      0.972     0.833     0.886        20
we